# Library

In [1937]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Data Loading

In [1938]:
df = pd.read_csv('laptop_price.csv', encoding='latin-1')
df

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,1316,Lenovo,Yoga 500-14ISK,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.8kg,638.00
1299,1317,Lenovo,Yoga 900-13ISK,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.3kg,1499.00
1300,1318,Lenovo,IdeaPad 100S-14IBR,Notebook,14.0,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2GB,64GB Flash Storage,Intel HD Graphics,Windows 10,1.5kg,229.00
1301,1319,HP,15-AC110nv (i7-6500U/6GB/1TB/Radeon,Notebook,15.6,1366x768,Intel Core i7 6500U 2.5GHz,6GB,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19kg,764.00


In [1939]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Memory            1303 non-null   object 
 9   Gpu               1303 non-null   object 
 10  OpSys             1303 non-null   object 
 11  Weight            1303 non-null   object 
 12  Price_euros       1303 non-null   float64
dtypes: float64(2), int64(1), object(10)
memory usage: 132.5+ KB


# Feature Engineering

## Vectorization of categorical columns

#### RAM

In [1940]:
df['Ram_GB'] = df['Ram'].str.replace('GB','').astype(int)

#### Weight

In [1941]:
df['Weight_kg'] = df['Weight'].str.replace('kg','').astype(float)

#### Storage / Memory

In [1942]:
def memory_parse(memory_str):
    total = 0
    parts = memory_str.split('+')

    for part in parts:
        part = part.strip()

        if 'TB' in part:
            num = float(part.split('TB')[0])
            total += num*1024 #Convert ke GB

        elif 'GB' in part:
            num = float(part.split('GB')[0])
            total += num

    return total

df['Storage_GB'] = df['Memory'].apply(memory_parse)

In [1943]:
# Create simplified storage type
def get_storage_type(memory):
    if 'SSD' in memory and 'HDD' in memory:
        return 'SSD + HDD'
    elif 'SSD' in memory:
        return 'SSD'
    elif 'HDD' in memory:
        return 'HDD'
    elif 'Flash Storage' in memory:
        return 'Flash Storage'
    elif 'Hybrid' in memory:
        return 'Hybrid'
    else:
        return 'Other Storage'

df['Storage_Type'] = df['Memory'].apply(get_storage_type)

#### Screen size

In [1944]:
df['Inches'] = df['Inches'].astype(float)

### GPU

In [1945]:
def get_gpu_brand(gpu):
    if 'Nvidia' in gpu :
        return 'Nvidia'
    elif 'AMD' in gpu:
        return 'AMD'
    else:
        return 'Intel'

df['Gpu_Brand'] = df['Gpu'].apply(get_gpu_brand)

In [1946]:
def get_gpu_type(gpu_type):
    if any (x in gpu_type for x in ['GTX', 'RTX', 'Quadro']):
        return 'Nvidia_Dedicated'
    elif any (x in gpu_type for x in ['GeForce', 'MX']):
        return 'Nvidia_Entry'
    elif any (x in gpu_type for x in ['Radeon RX', 'FirePro', 'Radeon Pro']):
        return 'AMD_Dedicated'
    elif 'AMD' in gpu_type:
        return 'AMD_Integrated'
    else:
        return 'Intel_Integrated'
    
df['Gpu_Type'] = df['Gpu'].apply(get_gpu_type)

#### Screen Resolution

In [1947]:
def get_resolution(res_str):
    if '3840' in res_str:
        return '4K'
    elif '3200' in res_str or '2880' in res_str or '2736' in res_str:
        return 'QHD'
    elif '2560' in res_str or '2304' in res_str or '2400' in res_str or '2256' in res_str or '2160' in res_str:
        return 'FHD+'
    elif '1920' in res_str or '1600' in res_str:
        return 'FHD'
    else:
        return 'HD'
    
df['Resolution'] = df['ScreenResolution'].apply(get_resolution)

#### CPU

In [1948]:
def get_cpu_brand(cpu_str):
    if 'Intel' in cpu_str:
        return 'Intel'
    elif 'AMD' in cpu_str:
        return 'AMD'
    else:
        return 'Other'
    
def get_cpu_tier(cpu_str):
    if 'i7' in cpu_str or 'i9' in cpu_str or 'Ryzen 7' in cpu_str or 'Ryzen 9' in cpu_str:
        return 'High'
    elif 'i5' in cpu_str or 'Ryzen 5' in cpu_str:
        return 'Mid'
    elif 'i3' in cpu_str or 'Ryzen 3' in cpu_str:
        return 'Entry'
    else:
        return 'Other'

df['Cpu_Brand'] = df['Cpu'].apply(get_cpu_brand)
df['Cpu_Tier']  = df['Cpu'].apply(get_cpu_tier)

#### Screen (Touchscreen & IPS panel)

In [1949]:
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

In [1950]:
df['IPS_Panel'] = df['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)

#### Price 

In [1951]:
# Asumsi Euro pada saat dataset
EURO_TO_IDR = 17500

df['Price_IDR'] = df['Price_euros'] * EURO_TO_IDR

# Create price segment using quartiles
df['Price_Segment'] = pd.qcut(
    df['Price_IDR'],
    q=4,
    labels=['Budget', 'Mid-Range', 'Premium', 'High-End']
)

In [1952]:
df[['Memory', 'Storage_GB', 'Ram', 'Ram_GB', 'Weight', 'Weight_kg', 'Gpu', 'Gpu_Brand', 'ScreenResolution','Resolution', 'Touchscreen', 'Cpu', 'Cpu_Brand', 'Cpu_Tier', 'Price_IDR', 'Price_Segment']].head(25)

,Memory,Storage_GB,Ram,Ram_GB,Weight,Weight_kg,Gpu,Gpu_Brand,ScreenResolution,Resolution,Touchscreen,Cpu,Cpu_Brand,Cpu_Tier,Price_IDR,Price_Segment
0,128GB SSD,128.0,8GB,8,1.37kg,1.37,Intel Iris Plus Graphics 640,Intel,IPS Panel Retina Display 2560x1600,FHD+,0,Intel Core i5 2.3GHz,Intel,Mid,23444575.0,Premium
1,128GB Flash Storage,128.0,8GB,8,1.34kg,1.34,Intel HD Graphics 6000,Intel,1440x900,HD,0,Intel Core i5 1.8GHz,Intel,Mid,15731450.0,Mid-Range
2,256GB SSD,256.0,8GB,8,1.86kg,1.86,Intel HD Graphics 620,Intel,Full HD 1920x1080,FHD,0,Intel Core i5 7200U 2.5GHz,Intel,Mid,10062500.0,Budget
3,512GB SSD,512.0,16GB,16,1.83kg,1.83,AMD Radeon Pro 455,AMD,IPS Panel Retina Display 2880x1800,QHD,0,Intel Core i7 2.7GHz,Intel,High,44405375.0,High-End
4,256GB SSD,256.0,8GB,8,1.37kg,1.37,Intel Iris Plus Graphics 650,Intel,IPS Panel Retina Display 2560x1600,FHD+,0,Intel Core i5 3.1GHz,Intel,Mid,31563000.0,High-End
5,500GB HDD,500.0,4GB,4,2.1kg,2.10,AMD Radeon R5,AMD,1366x768,HD,0,AMD A9-Series 9420 3GHz,AMD,Other,7000000.0,Budget
6,256GB Flash Storage,256.0,16GB,16,2.04kg,2.04,Intel Iris Pro Graphics,Intel,IPS Panel Retina Display 2880x1800,QHD,0,Intel Core i7 2.2GHz,Intel,High,37449475.0,High-End
7,256GB Flash Storage,256.0,8GB,8,1.34kg,1.34,Intel HD Graphics 6000,Intel,1440x900,HD,0,Intel Core i5 1.8GHz,Intel,Mid,20277250.0,Premium
8,512GB SSD,512.0,16GB,16,1.3kg,1.30,Nvidia GeForce MX150,Nvidia,Full HD 1920x1080,FHD,0,Intel Core i7 8550U 1.8GHz,Intel,High,26162500.0,High-End
9,256GB SSD,256.0,8GB,8,1.6kg,1.60,Intel UHD Graphics 620,Intel,IPS Panel Full HD 1920x1080,FHD,0,Intel Core i5 8250U 1.6GHz,Intel,Mid,13475000.0,Mid-Range


#### Feature Scaling

In [1953]:
scaler = MinMaxScaler()

df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price']] = scaler.fit_transform(df[['Ram_GB', 'Storage_GB', 'Weight_kg', 'Price_IDR']])

#### Encode


In [1954]:
type_encode = pd.get_dummies(df['TypeName'],prefix='type')
gpu_encode = pd.get_dummies(df['Gpu_Brand'],prefix='gpu')
resolution_encode = pd.get_dummies(df['Resolution'], prefix='res')
cpu_brand_encode = pd.get_dummies(df['Cpu_Brand'],  prefix='cpu')
cpu_tier_encode   = pd.get_dummies(df['Cpu_Tier'],   prefix='tier')

## Vector

In [1955]:
laptop_vector = pd.concat([df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Touchscreen']],
                           type_encode, 
                           gpu_encode,
                           resolution_encode,
                           cpu_brand_encode,
                           cpu_tier_encode
                           ], axis=1)

laptop_vector.index = df['laptop_ID']

In [1956]:
laptop_vector

,Norm_Ram,Norm_Storage,Norm_Weight,Norm_Price,Touchscreen,type_2 in 1 Convertible,type_Gaming,type_Netbook,type_Notebook,type_Ultrabook,...,res_FHD+,res_HD,res_QHD,cpu_AMD,cpu_Intel,cpu_Other,tier_Entry,tier_High,tier_Mid,tier_Other
laptop_ID,,,,,,,,,,,,,,,,,,,,,
1,0.096774,0.047022,0.169576,0.196741,0,False,False,False,False,True,...,True,False,False,False,True,False,False,False,True,False
2,0.096774,0.047022,0.162095,0.122353,0,False,False,False,False,True,...,False,True,False,False,True,False,False,False,True,False
3,0.096774,0.097179,0.291771,0.067679,0,False,False,False,True,False,...,False,False,False,False,True,False,False,False,True,False
4,0.225806,0.197492,0.284289,0.398895,0,False,False,False,False,True,...,False,False,True,False,True,False,False,True,False,False
5,0.096774,0.097179,0.169576,0.275038,0,False,False,False,False,True,...,True,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1316,0.032258,0.047022,0.276808,0.078312,1,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
1317,0.225806,0.197492,0.152120,0.223629,1,True,False,False,False,False,...,False,False,True,False,True,False,False,True,False,False
1318,0.000000,0.021944,0.201995,0.009283,0,False,False,False,True,False,...,False,True,False,False,True,False,False,False,False,True


### Cosine similarity

In [1957]:
def cosine_sim(vect1,vect2):
  norm_1 = np.linalg.norm(vect1)
  norm_2 = np.linalg.norm(vect2)

  cos_sim = (vect1 @ vect2) / (norm_1 * norm_2)
  return cos_sim


# Recommender System 1 - Specific input

In [1958]:
def recsys(type_name, gpu_brand, ram_user, storage_user, weight_user, price_user, resolution, touchscreen, cpu_brand, cpu_tier, top_N=5):
    
    # Toleransi budget yang dimunculkan 20% dari input
    budget_tolerance = 1.2
    filtered_df = df[df['Price_IDR'] <= price_user * budget_tolerance]
    filtered_vector = laptop_vector[laptop_vector.index.isin(filtered_df['laptop_ID'])]

    # Input User
    user_input = pd.DataFrame([[ram_user, storage_user, weight_user, price_user]],
                              columns=['Ram_GB', 'Storage_GB', 'Weight_kg', 'Price_IDR'])
    user_norm = scaler.transform(user_input)[0]

    # Query vector

    cossim = pd.Series(0.0,index=laptop_vector.columns)
    cossim[f'type_{type_name}'] = 1.0
    cossim[f'gpu_{gpu_brand}'] = 1.0
    cossim[f'res_{resolution}'] = 1.0    
    cossim[f'cpu_{cpu_brand}'] = 1.0
    cossim[f'tier_{cpu_tier}'] = 1.0
    cossim['Touchscreen']   = touchscreen
    cossim['Norm_Ram']      = user_norm[0]
    cossim['Norm_Storage']  = user_norm[1]
    cossim['Norm_Weight']   = user_norm[2]
    cossim['Norm_Price']    = user_norm[3]

    # Cosine similarity calculation

    score = [cosine_sim(cossim.values, x) for x in filtered_vector.values]
    result = pd.Series(score, index=filtered_vector.index)
    top_result = result.sort_values(ascending=False).head(top_N)

    # Result
    result_df = df[df['laptop_ID'].isin(top_result.index)][
        ['Product', 'Company', 'TypeName', 'Ram', 'Cpu', 'Gpu', 'Memory', 'ScreenResolution', 'Inches', 'Weight', 'Price_IDR']
    ].copy()
    result_df['Similarity'] = top_result.values
    result_df = result_df.sort_values('Similarity', ascending=False).reset_index(drop=True)

    return result_df

### Model Inference

Mencoba testing reccomender system dengan kriteria
- type_name = Gaming
- gpu_brand = Intel
- ram_gb = 32
- storage_gb = 512
- weight_kg = 1.5
- price_euros = 1200
- resolution= 'FHD',
- touchscreen= 1,
- cpu_brand= 'Intel',
- cpu_tier= 'High'

In [1959]:
recsys(
    type_name= 'Gaming',
    gpu_brand= 'AMD',
    ram_user= 8,
    storage_user= 512,
    weight_user= 2.8,
    price_user= 45000000,
    resolution= 'FHD+',
    touchscreen= 0,
    cpu_brand= 'Intel',
    cpu_tier= 'High', 
)

,Product,Company,TypeName,Ram,Cpu,Gpu,Memory,ScreenResolution,Inches,Weight,Price_IDR,Similarity
0,Alienware 17,Dell,Gaming,16GB,Intel Core i7 7820HK 2.9GHz,Nvidia GeForce GTX 1070,256GB SSD + 1TB HDD,IPS Panel 2560x1440,17.3,4.42kg,48982500.0,0.813263
1,Alienware 17,Dell,Gaming,16GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1070,128GB SSD + 1TB HDD,IPS Panel Full HD 1920x1080,17.3,4.42kg,52723475.0,0.810474
2,Alienware 17,Dell,Gaming,16GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1070,256GB SSD + 1TB HDD,IPS Panel 4K Ultra HD 3840x2160,15.6,4.42kg,50207325.0,0.807190
3,Omen 17-an012dx,HP,Gaming,12GB,Intel Core i7 7700HQ 2.8GHz,AMD Radeon RX 580,1TB HDD,IPS Panel Full HD 1920x1080,17.3,3.74kg,30607500.0,0.642750
4,Alienware 17,Dell,Gaming,16GB,Intel Core i7 6820HK 2.7GHz,Nvidia GeForce GTX 1080,512GB SSD + 1TB HDD,2560x1440,17.3,4.36kg,48265000.0,0.641512


# RECOMMENDER SYSTEM 2 - GENERAL SPECIFICATION


Berdasarkan hasil analisis, strategi recommendation engine (laptop) sebaiknya dibuat berdasarkan kebutuhan pengguna, bukan hanya berdasarkan brand atau harga. Setiap kategori pengguna memiliki prioritas yang berbeda, sehingga fitur rekomendasi perlu disesuaikan dengan tujuan penggunaan, budget, performa, portabilitas, dan kualitas display.

| User Segment | Kebutuhan Utama | Strategi Rekomendasi |
|---|---|---|
| Student / Basic User | Belajar, browsing, meeting online, dan produktivitas ringan | Prioritaskan laptop dengan `Price_Segment` Low sampai Mid-Low, RAM 4–8GB, integrated GPU, dan bobot yang masih mudah dibawa. |
| Office / Business User | Pekerjaan kantor, multitasking, dokumen, dan produktivitas harian | Prioritaskan laptop dengan RAM minimal 8GB, storage SSD, CPU menengah ke atas, ukuran layar nyaman, dan berat yang masih portable. |
| Gaming User | Gaming dan kebutuhan grafis tinggi | Prioritaskan laptop dengan GPU khusus/kartu grafis khusus, RAM besar, CPU kuat, layar lebih besar, serta SSD atau SSD + HDD. |
| Creative / Design User | Desain, editing, multimedia, dan pekerjaan visual | Prioritaskan laptop dengan `IPS_Panel`, GPU khusus/kartu grafis khusus, RAM tinggi, storage SSD, dan CPU yang kuat. |
| On-the-Go / Remote Worker User | Kerja mobile, sering bepergian, dan butuh laptop praktis | Prioritaskan laptop yang ringan, mudah dibawa, tipe `Ultrabook` atau `Netbook`, serta ukuran layar kecil hingga sedang. |
| Professional User | Pekerjaan data, programming, komputasi berat, dan multitasking intensif | Prioritaskan laptop dengan RAM 16GB atau lebih, CPU kuat seperti Intel Core i7 atau AMD Ryzen, SSD atau SSD + HDD, dan GPU khusus jika dibutuhkan. |
| Daily / General User | Penggunaan harian umum yang tidak terlalu spesifik | Prioritaskan laptop dengan spesifikasi seimbang, harga menengah, RAM cukup, storage yang nyaman, dan bobot yang masih praktis. |

Recommender System 2 ini bertujuan untuk membuat recommendation engine yang merujuk kepada user yang lebih general/awam. Dimana user tidak perlu memasukkan kebutuhan laptop yang spesifik untuk ditentukan rekomendasinya oleh system.

User hanya perlu memasukkan :
    
    1. Kebutuhan / User Segment

    2. Berat laptop yang di inginkan = Ringan, Sedang, Berat

    3. Harga Laptop  = Budget, Mid-Range, Premium, High-End

#### Scaling 


Dibutuhkan scaling tambahan terhadap kolom inches karena belum dilakukan di recommender system sebelumnya.

In [1960]:
inches_scaler = MinMaxScaler()
df['Norm_Inches'] = inches_scaler.fit_transform(df[['Inches']])

In [1961]:
df['Affordability'] = 1 - df['Norm_Price']

#### Encode

Encoding tambahan pada GPU_type dan Storage dikarenakan belum dibutuhkan di recommender system sebelumnya

In [1962]:
type_encode = pd.get_dummies(df['TypeName'],prefix='type')
gpu_type_encode = pd.get_dummies(df['Gpu_Type'],prefix='gpu_type')
cpu_tier_encode   = pd.get_dummies(df['Cpu_Tier'],   prefix='tier')
storage_encode = pd.get_dummies(df['Storage_Type'], prefix = 'storage')

#### Vector 

In [1963]:
laptop_vector2 = pd.concat([df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Affordability', 'Norm_Inches', 'IPS_Panel']],
                           type_encode, 
                           gpu_type_encode,
                           cpu_tier_encode,
                           storage_encode
                           ], axis=1)

laptop_vector2.index = df['laptop_ID']

print(laptop_vector2.columns.tolist())


['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Affordability', 'Norm_Inches', 'IPS_Panel', 'type_2 in 1 Convertible', 'type_Gaming', 'type_Netbook', 'type_Notebook', 'type_Ultrabook', 'type_Workstation', 'gpu_type_AMD_Dedicated', 'gpu_type_AMD_Integrated', 'gpu_type_Intel_Integrated', 'gpu_type_Nvidia_Dedicated', 'gpu_type_Nvidia_Entry', 'tier_Entry', 'tier_High', 'tier_Mid', 'tier_Other', 'storage_Flash Storage', 'storage_HDD', 'storage_Hybrid', 'storage_SSD', 'storage_SSD + HDD']


#### Mapping User Segment & Weighted Cosine Similarity

Dibuatlah profile sesuai user segment yang dimana sesuai dengan Strategi rekomendasi.

In [1964]:
# Profile

weights = {
    'Student / Basic User': { #harga penting, mudah dibawa, 4-8 gb ram cukup
        'Affordability'             : 1.0, 
        'Norm_Weight'               : 0.8,  
        'Norm_Ram'                  : 0.5, 
        'Norm_Storage'              : 0.4,
        'Norm_Inches'               : 0.4,
        'IPS_Panel'                 : 0.2,
        'type_Netbook'              : 0.8,
        'type_Notebook'             : 0.7,
        'type_Ultrabook'            : 0.6,
        'gpu_type_AMD_Integrated'   : 0.8,  
        'gpu_type_Intel_Integrated' : 0.8,
        'tier_Entry'                : 0.7,
        'tier_Mid'                  : 0.5,
        'storage_HDD'               : 0.5,
        'storage_SSD'               : 0.6,
    },
    'Office / Business User': { #portable, layar nyaman, cpu mid to high, ssd penting
        'Norm_Ram'                  : 0.9,  
        'Norm_Weight'               : 0.7, 
        'Affordability'             : 0.6,
        'Norm_Storage'              : 0.7,
        'Norm_Inches'               : 0.6,
        'IPS_Panel'                 : 0.5,
        'type_Ultrabook'            : 0.9,
        'type_Notebook'             : 0.7,
        'tier_High'                 : 0.9,
        'tier_Mid'                  : 0.7,
        'gpu_type_Intel_Integrated' : 0.6,
        'storage_SSD'               : 1.0,  
    },
    'Gaming User': { #ram besar, layar besar, type gaming, gpu dedicated, cpu kuat, ssd + hdd
        'Norm_Ram'                  : 0.9,  
        'Norm_Storage'              : 0.8,
        'Norm_Inches'               : 0.8, 
        'Norm_Weight'               : 0.3,
        'Affordability'             : 0.5,
        'IPS_Panel'                 : 0.5,
        'type_Gaming'               : 1.0,  
        'gpu_type_Nvidia_Dedicated' : 1.0, 
        'gpu_type_AMD_Dedicated'    : 0.8,
        'tier_High'                 : 1.0,  
        'tier_Mid'                  : 0.5,
        'storage_SSD'               : 0.8,
        'storage_SSD + HDD'         : 0.9,  
    },
    'Creative / Design User': { #ram tinggi, layar ips, cpu tinggi untuk processing
        'Norm_Ram'                  : 1.0,  
        'Norm_Storage'              : 0.8,
        'Norm_Inches'               : 0.7,
        'Norm_Weight'               : 0.4,
        'Affordability'             : 0.5,
        'IPS_Panel'                 : 1.0,  
        'type_Workstation'          : 0.9,
        'gpu_type_Nvidia_Dedicated' : 1.0,  
        'gpu_type_AMD_Dedicated'    : 0.9,
        'tier_High'                 : 1.0,  
        'storage_SSD'               : 0.9,
    },
    'On-the-Go / Remote Worker': { #ringan, layar kecil
        'Norm_Weight'               : 1.0,  
        'Norm_Inches'               : 0.8, 
        'Affordability'             : 0.6,
        'Norm_Ram'                  : 0.5,
        'Norm_Storage'              : 0.4,
        'IPS_Panel'                 : 0.4,
        'type_Ultrabook'            : 1.0, 
        'type_Netbook'              : 0.8,
        'gpu_type_Intel_Integrated' : 0.6,
        'tier_Mid'                  : 0.7,
        'storage_SSD'               : 0.7,
    },
    'Professional User': { # ram besar, gpu dedicated, cpu tinggi
        'Norm_Ram'                  : 1.0,  
        'Norm_Storage'              : 0.9,
        'Affordability'             : 0.5,
        'Norm_Weight'               : 0.4,
        'Norm_Inches'               : 0.6,
        'IPS_Panel'                 : 0.7,
        'type_Workstation'          : 1.0,
        'gpu_type_Nvidia_Dedicated' : 0.8,  
        'gpu_type_AMD_Dedicated'    : 0.7,
        'tier_High'                 : 1.0,  
        'storage_SSD'               : 0.8,
        'storage_SSD + HDD'         : 0.9,
    },
    'Daily / General User': { # seimbang all around
        'Norm_Ram'                  : 0.7, 
        'Norm_Storage'              : 0.7,
        'Norm_Weight'               : 0.7,
        'Affordability'             : 0.8,
        'Norm_Inches'               : 0.6,
        'IPS_Panel'                 : 0.5,
        'type_Notebook'             : 0.9,
        'type_Ultrabook'            : 0.7,
        'gpu_type_Intel_Integrated' : 0.6,
        'gpu_type_AMD_Integrated'   : 0.6,
        'tier_Mid'                  : 0.9, 
        'storage_SSD'               : 0.7,
    }
}

#### Weighted Cosine Similarity

In [1965]:
# Weighted Cosine Sim

def weighted_cosine_sim(vect1,vect2,w):
  vect1_w = vect1 *w
  vect2_w = vect2 * w
  norm_1 = np.linalg.norm(vect1_w)
  norm_2 = np.linalg.norm(vect2_w)

  if norm_1 == 0 or norm_2 == 0:
    return 0.0  
  w_cos_sim = (vect1_w @ vect2_w) / (norm_1 * norm_2)
  return w_cos_sim

# Mapping berat & Budget
weight_mapping = {
  'Ringan'  : 0.0,
  'Sedang'  : 0.5,
  'Berat'   : 1.0
}
budget_mapping = {
  'Budget'       : 1.0,
  'Mid-Range'    : 0.75,
  'Premium'      : 0.25,
  'High-End'     : 0.0
}


#### Query Default


Query default adalah profile ideal untuk setiap user segment. Kalau tidak ada query default, Query vector hanya berisi berat dan budget dan sisanya akan 0 dan menghasilkan cosine similarity yang rendah. System tidak tau kalau profile A ingin punya laptop yang ideal itu bagaimana. 

Dengan adanya Query Default, vector jadi lebih lengkap dan lebih banyak similarity antara query dan dataset.

Query default disesuaikan dengan Strategi Rekomendasi.

In [1966]:
query_defaults = {
    'Student / Basic User': {
        'type_Notebook'             : 1.0,
        'gpu_type_Intel_Integrated' : 1.0,
        'gpu_type_AMD_Integrated'   : 0.8,
        'tier_Entry'                : 1.0,
        'tier_Mid'                  : 0.5,
        'storage_HDD'               : 1.0,
        'storage_SSD'               : 0.7,
    },
    'Office / Business User': {
        'type_Ultrabook'            : 1.0,
        'type_Notebook'             : 0.7,
        'Norm_Inches'               : 0.7,
        'gpu_type_AMD_Integrated'   : 1.0,
        'gpu_type_Intel_Integrated' : 0.8,
        'tier_High'                 : 1.0,
        'tier_Mid'                  : 0.8,
        'storage_SSD'               : 1.0,
    },
    'Gaming User': {
        'type_Gaming'               : 1.0,
        'Norm_Inches'               : 0.8,
        'gpu_type_Nvidia_Dedicated' : 1.0,
        'gpu_type_AMD_Dedicated'    : 0.8,
        'tier_High'                 : 1.0,
        'storage_SSD'               : 0.8,
        'storage_SSD + HDD'         : 1.0,
    },
    'Creative / Design User': {
        'type_Workstation'          : 1.0,
        'gpu_type_Nvidia_Dedicated' : 1.0,
        'gpu_type_AMD_Dedicated'    : 0.9,
        'tier_High'                 : 1.0,
        'storage_SSD'               : 1.0,
        'IPS_Panel'                 : 1.0,
    },
    'On-the-Go / Remote Worker': {
        'type_Ultrabook'            : 1.0,
        'type_Netbook'              : 1.0,
        'gpu_type_Intel_Integrated' : 1.0,
        'gpu_type_AMD_Integrated'   : 0.7,
        'tier_Mid'                  : 1.0,
        'tier_Entry'                : 0.7,
        'storage_SSD'               : 1.0,
    },
    'Professional User': {
        'type_Workstation'          : 1.0,
        'gpu_type_Nvidia_Dedicated' : 0.8,
        'gpu_type_AMD_Dedicated'    : 0.7,
        'tier_High'                 : 1.0, 
        'storage_SSD'               : 0.8,
        'storage_SSD + HDD'         : 1.0,
        'Norm_Ram'                  : 1.0,  
    },
    'Daily / General User': {
        'type_Notebook'             : 1.0,
        'type_Ultrabook'            : 0.7,
        'gpu_type_Intel_Integrated' : 0.8,
        'gpu_type_AMD_Integrated'   : 0.8,
        'tier_Mid'                  : 1.0,
        'storage_SSD'               : 0.8,
        'storage_HDD'               : 0.6,
    },
}

### Recommender System 2

In [1967]:
def recsys2(user_need, berat, budget, top_N=5):

    # Filter sesuai price segment
    filtered_df = df[df['Price_Segment'] == budget]
    filtered_vector = laptop_vector2[laptop_vector2.index.isin(filtered_df['laptop_ID'])]

    # Query Vector
    cossim = pd.Series(0.0, index=laptop_vector2.columns)
    cossim['Norm_Weight'] = weight_mapping[berat]
    cossim['Affordability'] = budget_mapping[budget]

    for col, val in query_defaults[user_need].items():
        if col in cossim.index:
            cossim[col] = val

    # Weight
    w = pd.Series(0.1, index=laptop_vector2.columns)
    for col, val in weights[user_need].items():
        if col in w.index:
            w[col] = val

    # Cossim hanya dengan price segmentnya saja
    score = [weighted_cosine_sim(cossim.values, x, w.values) for x in filtered_vector.values]
    result = pd.Series(score, index=filtered_vector.index)
    top_result = result.sort_values(ascending=False).head(top_N)

    # Result
    result_df = df[df['laptop_ID'].isin(top_result.index)][
        ['Product', 'Company', 'TypeName', 'Ram', 'Cpu', 'Gpu', 'Memory', 'ScreenResolution', 'Inches', 'Weight', 'Price_IDR']
    ].copy()
    result_df['Similarity'] = top_result.values
    result_df = result_df.sort_values('Similarity', ascending=False).reset_index(drop=True)

    return result_df

#### Inference

In [1968]:
recsys2('Gaming User',berat='Berat', budget='Mid-Range')

,Product,Company,TypeName,Ram,Cpu,Gpu,Memory,ScreenResolution,Inches,Weight,Price_IDR,Similarity
0,FX753VE-GC093 (i7-7700HQ/12GB/1TB/GeForce,Asus,Gaming,12GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050 Ti,1TB HDD,Full HD 1920x1080,17.3,3kg,16607500.0,0.873498
1,Legion Y520-15IKBN,Lenovo,Gaming,8GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050M,256GB SSD,IPS Panel Full HD 1920x1080,15.6,2.4kg,15207500.0,0.842559
2,Inspiron 7567,Dell,Gaming,8GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050,1.0TB Hybrid,Full HD 1920x1080,15.6,2.62kg,15732500.0,0.814152
3,Inspiron 7567,Dell,Gaming,8GB,Intel Core i7 7700HQ 2.8GHz,Nvidia GeForce GTX 1050Ti,1TB HDD,Full HD 1920x1080,15.6,2.62kg,16257500.0,0.812912
4,Rog GL552VW-DM201T,Asus,Gaming,8GB,Intel Core i7 6700HQ 2.6GHz,Nvidia GeForce GTX 960M,256GB SSD + 1TB HDD,IPS Panel Full HD 1920x1080,15.6,2.591kg,15907500.0,0.812882


In [1969]:
recsys2('Student / Basic User',berat='Sedang', budget='Mid-Range')

,Product,Company,TypeName,Ram,Cpu,Gpu,Memory,ScreenResolution,Inches,Weight,Price_IDR,Similarity
0,ProBook 450,HP,Notebook,4GB,Intel Core i3 7100U 2.4GHz,Intel HD Graphics 620,500GB HDD,1366x768,15.6,2.1kg,11698400.0,0.879137
1,Latitude 3380,Dell,Notebook,4GB,Intel Core i3 6006U 2GHz,Intel HD Graphics 520,500GB HDD,1366x768,13.3,1.65kg,14857500.0,0.877909
2,Probook 440,HP,Notebook,4GB,Intel Core i3 7100U 2.4GHz,Intel HD Graphics 620,500GB HDD,1366x768,14.0,1.64kg,11970000.0,0.876363
3,Probook 430,HP,Notebook,4GB,Intel Core i3 7100U 2.4GHz,Intel HD Graphics 620,500GB HDD,1366x768,13.3,1.49kg,14000000.0,0.876160
4,ProBook 650,HP,Notebook,4GB,Intel Core i3 6100U 2.3GHz,Intel HD Graphics 520,500GB HDD,1366x768,15.6,2.31kg,12340125.0,0.876079


In [1970]:
recsys2('Professional User',berat='Ringan', budget='High-End')

,Product,Company,TypeName,Ram,Cpu,Gpu,Memory,ScreenResolution,Inches,Weight,Price_IDR,Similarity
0,Precision 7720,Dell,Workstation,16GB,Intel Core i7 7820HQ 2.9GHz,Nvidia Quadro M1200,256GB SSD,Full HD 1920x1080,17.3,3.42kg,50485050.0,0.772890
1,Precision M5520,Dell,Workstation,8GB,Intel Core i7 7700HQ 2.8GHz,Nvidia Quadro M1200,256GB SSD,4K Ultra HD / Touchscreen 3840x2160,15.6,1.78kg,47460000.0,0.768570
2,Precision M5520,Dell,Workstation,8GB,Intel Core i7 7700HQ 2.8GHz,Nvidia Quadro M1200,256GB SSD,Full HD 1920x1080,15.6,1.78kg,42140000.0,0.759569
3,ThinkPad P51s,Lenovo,Workstation,16GB,Intel Core i7 6500U 2.5GHz,Nvidia Quadro M520M,512GB SSD,Full HD 1920x1080,15.6,2.18kg,32462500.0,0.750564
4,ZBook 15,HP,Workstation,16GB,Intel Core i7 7700HQ 2.8GHz,Nvidia Quadro M2200,256GB SSD,Full HD 1920x1080,15.6,2.6kg,42332500.0,0.748993


In [1971]:
recsys2('On-the-Go / Remote Worker', berat='Ringan', budget='Mid-Range')

,Product,Company,TypeName,Ram,Cpu,Gpu,Memory,ScreenResolution,Inches,Weight,Price_IDR,Similarity
0,Swift 3,Acer,Ultrabook,8GB,Intel Core i5 8250U 1.6GHz,Intel UHD Graphics 620,256GB SSD,IPS Panel Full HD 1920x1080,14.0,1.6kg,13475000.0,0.870020
1,Vostro 5471,Dell,Ultrabook,8GB,Intel Core i5 8250U 1.6GHz,Intel UHD Graphics 620,256GB SSD,Full HD 1920x1080,14.0,1.7kg,15382500.0,0.857941
2,Vostro 5370,Dell,Ultrabook,8GB,Intel Core i5 8250U 1.6GHz,Intel UHD Graphics 620,256GB SSD,Full HD 1920x1080,13.3,1.41kg,16607500.0,0.847292
3,XPS 13,Dell,Ultrabook,8GB,Intel Core i5 7200U 2.5GHz,Intel HD Graphics 620,128GB SSD,IPS Panel Full HD / Touchscreen 1920x1080,13.3,1.29kg,15732500.0,0.836291
4,Swift 3,Acer,Ultrabook,8GB,Intel Core i5 7200U 2.5GHz,Intel Graphics 620,256GB SSD,IPS Panel Full HD 1920x1080,14.0,1.8kg,16082500.0,0.832822
